# LAK-3 — Snapshot growth & expiration

**Break → Detect → Fix → Prove.** Every write to an Iceberg table — `append`, `overwrite`,
`MERGE` — creates a **new snapshot**. Snapshots power time-travel and rollback, but they are
kept *forever* unless you expire them. On a busy table the snapshot history (and the
metadata-file log) grows **unbounded**: metadata bloats, planning slows, and old snapshots
**pin old data files** so storage never shrinks.

**Pre-requisite:** the unified Spark server is up (`make up`) → Spark UI at http://localhost:4040.
This notebook connects via Spark Connect. **Laptop-safe:** one tiny table, ~20 small writes, all
under `.tmp/`; the Teardown cell drops the table (`make clean` clears everything).

See the companion [`README.md`](./README.md), the [Spark-UI guide](../../docs/spark-ui-guide.md),
and the [troubleshooting sheet](../../docs/troubleshooting.md). Headline metric: **snapshots**
from [`common.iceberg_meta`](../../common/iceberg_meta.py).

In [ ]:
from common.spark_session import spark, display_df
from common.iceberg_meta import table_health, compare_health
from pyspark.sql import functions as F

T = "iceberg_catalog.default.lak3_orders"   # namespace pre-created by docker-entrypoint.sh
spark

## Step 0 — a fresh Iceberg orders table

We start clean (drop if it exists) and create a tiny `orders` table with a single seed batch.
That `CREATE` is itself the **first snapshot**.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {T}")

seed = (spark.range(0, 100).withColumnRenamed("id", "order_id")
        .withColumn("customer_id", F.pmod(F.col("order_id"), F.lit(10)))
        .withColumn("amount", F.round(F.rand(seed=1) * 100, 2))
        .withColumn("status", F.lit("new")))
seed.writeTo(T).using("iceberg").create()                 # snapshot #1 (the create)

print("rows:", spark.table(T).count())
compare_health([table_health(spark, T, "after create")])   # snapshots = 1

## 1. Break it — ~20 writes, one snapshot each

A realistic busy table: a steady trickle of small `append`s plus the occasional `MERGE` (an
upsert / late correction) and an `overwrite` (a status reclassification). We do ~19 more writes.
**Nothing about the row count explodes — but the snapshot count climbs with every commit.**

> Why this is laptop-safe: each write touches at most a few rows, so the table stays tiny. The
> pathology here is *metadata* growth (snapshots + metadata files), not data volume — exactly the
> "shrink the box" trick: small data, real production failure mode.

In [ ]:
next_id = 100
for i in range(15):                       # 15 small appends
    batch = (spark.range(next_id, next_id + 5).withColumnRenamed("id", "order_id")
             .withColumn("customer_id", F.pmod(F.col("order_id"), F.lit(10)))
             .withColumn("amount", F.round(F.rand(seed=i) * 100, 2))
             .withColumn("status", F.lit("new")))
    batch.writeTo(T).append()
    next_id += 5

# 3 MERGEs (upserts: bump amount on a handful of existing orders)
for i in range(3):
    upd = spark.createDataFrame([(o, 999.0 + i) for o in range(i * 3, i * 3 + 3)],
                                ["order_id", "amount"])
    upd.createOrReplaceTempView("upd")
    spark.sql(f"""
        MERGE INTO {T} t USING upd u ON t.order_id = u.order_id
        WHEN MATCHED THEN UPDATE SET t.amount = u.amount
    """)

# 1 overwrite (reclassify everything to 'archived')
spark.sql(f"INSERT OVERWRITE {T} SELECT order_id, customer_id, amount, 'archived' AS status FROM {T}")

print("total rows:", spark.table(T).count())

## 2. Detect it — count the snapshots & the metadata log

Two Iceberg metadata tables tell the story (both read fine over Spark Connect):

- **`<t>.snapshots`** — one row per commit. `COUNT(*)` is the headline number; it should be ~20.
- **`<t>.metadata_log_entries`** — one row per `vN.metadata.json` file written. Every commit
  rewrites the table-metadata pointer, so this log grows alongside the snapshots (and is what
  bloats the metadata directory on disk).

In [ ]:
n_snapshots = spark.sql(f"SELECT COUNT(*) AS n FROM {T}.snapshots").first()["n"]
n_metalog   = spark.sql(f"SELECT COUNT(*) AS n FROM {T}.metadata_log_entries").first()["n"]
print(f"snapshots in history     : {n_snapshots}")
print(f"metadata_log_entries     : {n_metalog}")

before = table_health(spark, T, "before expire")   # capture for the Prove step
compare_health([before])                            # snapshots ~= 20

In [ ]:
print("snapshot history (operation per commit):")
spark.sql(f"""
    SELECT committed_at, snapshot_id, operation
    FROM {T}.snapshots ORDER BY committed_at
""").show(50, truncate=False)

print("metadata-file log (each commit rewrites the metadata pointer):")
spark.sql(f"SELECT timestamp, file FROM {T}.metadata_log_entries ORDER BY timestamp").show(5, truncate=False)

## 3. Diagnose

Iceberg keeps **every** snapshot so you can time-travel (`VERSION AS OF` / `TIMESTAMP AS OF`) or
roll back a bad write — that's a feature. But nothing prunes them automatically by default, so on
a high-frequency table they accumulate without bound. Two costs:

1. **Metadata bloat & slower planning.** Each commit appends a snapshot to the table metadata and
   writes a new `vN.metadata.json`. The metadata grows; query planning and commits get heavier.
2. **Storage that never shrinks.** An old snapshot **pins the data files it referenced** —
   `overwrite`/`MERGE` write *new* files but the superseded ones can't be deleted while a snapshot
   still points at them. Disk keeps growing even though the live table is tiny. (Reclaiming those
   files is `expire_snapshots`' job; truly orphaned files are LAK-4.)

## 4. Fix it — `expire_snapshots`

Drop old snapshots and let Iceberg delete the data files only they referenced.

**Gotcha (important):** `expire_snapshots` has a built-in age guard — by default it won't expire
snapshots younger than `history.expire.max-snapshot-age-ms` (**5 days**). Our snapshots were all
just created, so a bare call expires *nothing*. To expire **fresh** snapshots you must pass
`older_than => now()` (lift the age guard) and/or `retain_last => N` (keep only the newest N). We
keep the last **3** so a short time-travel window still works.

In [ ]:
RETAIN = 3
res = spark.sql(f"""
    CALL iceberg_catalog.system.expire_snapshots(
        table => 'default.lak3_orders',
        older_than => now(),     -- lift the 5-day age guard so fresh snapshots are eligible
        retain_last => {RETAIN}  -- but always keep the newest 3 (time-travel window)
    )
""")
res.show(truncate=False)   # returns counts of data/manifest/metadata files deleted

### Make it automatic — table properties

Calling the procedure by hand doesn't scale. Set retention **properties** on the table so expiry
behaves predictably (and so a scheduled `expire_snapshots` job — see *in production* below — keeps
the table healthy):

```sql
ALTER TABLE iceberg_catalog.default.lak3_orders SET TBLPROPERTIES (
    'history.expire.max-snapshot-age-ms' = '604800000',  -- 7 days: how long snapshots live
    'history.expire.min-snapshots-to-keep' = '3'          -- floor: never drop below 3
);
```

These don't expire on their own — Iceberg has no background thread — but they are the policy a
scheduled maintenance job (or engine auto-maintenance) reads. We set them below so the table
carries its own retention policy.

In [ ]:
spark.sql(f"""
    ALTER TABLE {T} SET TBLPROPERTIES (
        'history.expire.max-snapshot-age-ms' = '604800000',
        'history.expire.min-snapshots-to-keep' = '3'
    )
""")
print("retention properties set (7-day max age, keep >= 3 snapshots).")

## 5. Prove it

Before/after `table_health`. The **Snapshots** row collapses from ~20 to `retain_last` (3).
Data files drop too, because expiring the old snapshots unpinned the files that `overwrite`/`MERGE`
superseded.

In [ ]:
after = table_health(spark, T, "after expire")
compare_health([before, after])

n_after = spark.sql(f"SELECT COUNT(*) AS n FROM {T}.snapshots").first()["n"]
print(f"\nsnapshots: {before['snapshots']} -> {n_after}   (retain_last = {RETAIN})")
print("table still queryable:", spark.table(T).count(), "rows")

## Takeaways & "in real production…"

- **Detect** snapshot bloat with `SELECT COUNT(*) FROM <t>.snapshots` (and watch
  `<t>.metadata_log_entries` for metadata-file growth). Rising snapshot counts on a high-write
  table are the signal.
- **The age-guard gotcha:** `expire_snapshots` refuses to drop snapshots younger than the
  max-snapshot-age (5 days default). Pass `older_than => now()` and/or `retain_last => N` to expire
  fresh ones — exactly what bit us here at laptop scale.
- **Expiry reclaims storage**, not just metadata: it deletes data files that only the expired
  snapshots referenced (CoW/MERGE leftovers). Truly *orphaned* files (failed writes) are a separate
  job — **LAK-4** (`remove_orphan_files`).
- **In production:** **schedule** `expire_snapshots` (nightly/weekly) and set
  `history.expire.max-snapshot-age-ms` + `history.expire.min-snapshots-to-keep` so the policy is
  explicit. **Balance against the time-travel window you actually need** — expiring too aggressively
  destroys the snapshots rollback/audit depends on (**LAK-9** time-travel & rollback). Retention is a
  deliberate tradeoff: storage & planning cost vs. how far back you must be able to look.

## Teardown

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {T}")
print("Dropped lak3_orders. (`make clean` clears all generated data under .tmp/.)")